In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, load_iris, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.decomposition import PCA as SklearnPCA
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("="*80)
print("ALGORITHM 1: K-MEANS CLUSTERING")
print("="*80)

print("\n--- Approach 1: Standard K-Means from Scratch ---")

class KMeansScratch:
    def __init__(self, n_clusters=3, max_iters=100, tol=1e-4):
        self.n_clusters = n_clusters
        self.max_iters = max_iters
        self.tol = tol
        
    def fit(self, X):
        # Random initialization of centroids
        n_samples, n_features = X.shape
        random_indices = np.random.choice(n_samples, self.n_clusters, replace=False)
        self.centroids = X[random_indices]
        
        for _ in range(self.max_iters):
            # Assign clusters
            distances = self._compute_distances(X)
            self.labels = np.argmin(distances, axis=1)
            
            # Update centroids
            new_centroids = np.zeros_like(self.centroids)
            for k in range(self.n_clusters):
                if np.sum(self.labels == k) > 0:
                    new_centroids[k] = X[self.labels == k].mean(axis=0)
                else:
                    new_centroids[k] = self.centroids[k]
            
            # Check convergence
            if np.linalg.norm(new_centroids - self.centroids) < self.tol:
                break
                
            self.centroids = new_centroids
        
        return self
    
    def _compute_distances(self, X):
        distances = np.zeros((X.shape[0], self.n_clusters))
        for k in range(self.n_clusters):
            distances[:, k] = np.linalg.norm(X - self.centroids[k], axis=1)
        return distances
    
    def predict(self, X):
        distances = self._compute_distances(X)
        return np.argmin(distances, axis=1)

# Generate synthetic dataset
X_kmeans, y_kmeans = make_blobs(n_samples=300, n_features=2, centers=3, 
                                 cluster_std=0.60, random_state=42)

# Apply K-Means from scratch
kmeans_scratch = KMeansScratch(n_clusters=3)
kmeans_scratch.fit(X_kmeans)
predictions = kmeans_scratch.predict(X_kmeans)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X_kmeans[:, 0], X_kmeans[:, 1], c=y_kmeans, cmap='viridis', alpha=0.6)
axes[0].scatter(kmeans_scratch.centroids[:, 0], kmeans_scratch.centroids[:, 1], 
                c='red', marker='X', s=200, linewidth=3)
axes[0].set_title('True Labels with Centroids')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

axes[1].scatter(X_kmeans[:, 0], X_kmeans[:, 1], c=predictions, cmap='viridis', alpha=0.6)
axes[1].scatter(kmeans_scratch.centroids[:, 0], kmeans_scratch.centroids[:, 1], 
                c='red', marker='X', s=200, linewidth=3)
axes[1].set_title('K-Means Predictions (Scratch)')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
plt.tight_layout()
plt.show()

print("Dataset: Synthetic Blobs (300 samples, 2 features, 3 clusters)")
print(f"Centroids found: {kmeans_scratch.centroids}")

print("\n--- Approach 2: K-Means with Elbow Method & Mini-Batch ---")
from sklearn.cluster import MiniBatchKMeans, KMeans
from sklearn.metrics import silhouette_score

# Load Iris dataset
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# Standardize the data
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

# Elbow method to find optimal K
inertias = []
silhouette_scores = []
K_range = range(2, 10)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_iris_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_iris_scaled, kmeans.labels_))

# Find optimal K using elbow and silhouette
optimal_k_elbow = K_range[np.argmin(np.diff(inertias, 2)) + 1]
optimal_k_silhouette = K_range[np.argmax(silhouette_scores)]

print(f"Optimal K by Elbow Method: {optimal_k_elbow}")
print(f"Optimal K by Silhouette Score: {optimal_k_silhouette}")

# Apply Mini-Batch K-Means
mbkmeans = MiniBatchKMeans(n_clusters=3, batch_size=50, random_state=42, n_init=10)
mbkmeans.fit(X_iris_scaled)
mbkmeans_labels = mbkmeans.predict(X_iris_scaled)

# Visualize elbow curve
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(K_range, inertias, 'bo-')
axes[0].axvline(x=optimal_k_elbow, color='red', linestyle='--', label=f'Optimal K={optimal_k_elbow}')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method for Optimal K')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(K_range, silhouette_scores, 'go-')
axes[1].axvline(x=optimal_k_silhouette, color='red', linestyle='--', label=f'Best K={optimal_k_silhouette}')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score for Optimal K')
axes[1].legend()
axes[1].grid(True)
plt.tight_layout()
plt.show()

print(f"\nDataset: Iris (150 samples, 4 features, 3 true classes)")
print(f"Mini-Batch K-Means Inertia: {mbkmeans.inertia_:.2f}")
print(f"Silhouette Score: {silhouette_score(X_iris_scaled, mbkmeans_labels):.3f}")

In [ ]:
print("\n--- Approach 3: K-Means++ Initialization ---")

class KMeansPlusPlus:
    def __init__(self, n_clusters=3, max_iters=100, tol=1e-4):
        self.n_clusters = n_clusters
        self.max_iters = max_iters
        self.tol = tol
    
    def _initialize_centroids_plus_plus(self, X):
        n_samples = X.shape[0]
        centroids = []
        
        # Choose first centroid randomly
        centroids.append(X[np.random.choice(n_samples)])
        
        # Choose remaining centroids based on distance
        for _ in range(1, self.n_clusters):
            distances = np.min([np.linalg.norm(X - centroid, axis=1) for centroid in centroids], axis=0)
            probabilities = distances / distances.sum()
            cumulative_probabilities = np.cumsum(probabilities)
            r = np.random.random()
            
            for j, p in enumerate(cumulative_probabilities):
                if r < p:
                    centroids.append(X[j])
                    break
        
        return np.array(centroids)
    
    def fit(self, X):
        self.centroids = self._initialize_centroids_plus_plus(X)
        
        for _ in range(self.max_iters):
            distances = self._compute_distances(X)
            self.labels = np.argmin(distances, axis=1)
            
            new_centroids = np.zeros_like(self.centroids)
            for k in range(self.n_clusters):
                if np.sum(self.labels == k) > 0:
                    new_centroids[k] = X[self.labels == k].mean(axis=0)
                else:
                    new_centroids[k] = self.centroids[k]
            
            if np.linalg.norm(new_centroids - self.centroids) < self.tol:
                break
                
            self.centroids = new_centroids
        
        return self
    
    def _compute_distances(self, X):
        distances = np.zeros((X.shape[0], self.n_clusters))
        for k in range(self.n_clusters):
            distances[:, k] = np.linalg.norm(X - self.centroids[k], axis=1)
        return distances

# Generate dataset with uneven cluster sizes
X_uneven, _ = make_blobs(n_samples=500, n_features=2, centers=4, 
                          cluster_std=[0.4, 0.6, 0.8, 0.5], random_state=42)

# Compare standard vs K-Means++
kmeans_standard = KMeansScratch(n_clusters=4)
kmeans_standard.fit(X_uneven)

kmeans_pp = KMeansPlusPlus(n_clusters=4)
kmeans_pp.fit(X_uneven)

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X_uneven[:, 0], X_uneven[:, 1], c=kmeans_standard.labels, cmap='viridis', alpha=0.6)
axes[0].scatter(kmeans_standard.centroids[:, 0], kmeans_standard.centroids[:, 1], 
                c='red', marker='X', s=200, linewidth=3)
axes[0].set_title('Standard K-Means (Random Init)')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

axes[1].scatter(X_uneven[:, 0], X_uneven[:, 1], c=kmeans_pp.labels, cmap='viridis', alpha=0.6)
axes[1].scatter(kmeans_pp.centroids[:, 0], kmeans_pp.centroids[:, 1], 
                c='red', marker='X', s=200, linewidth=3)
axes[1].set_title('K-Means++ (Smart Initialization)')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
plt.tight_layout()
plt.show()

print("Dataset: Uneven Blobs (500 samples, 2 features, 4 clusters)")
print("K-Means++ provides better initialization, leading to more stable clustering")

In [ ]:
print("\n" + "="*80)
print("ALGORITHM 2: PRINCIPAL COMPONENT ANALYSIS (PCA)")
print("="*80)

# Load high-dimensional dataset
cancer = load_breast_cancer()
X_cancer = cancer.data
y_cancer = cancer.target
feature_names = cancer.feature_names

print(f"Dataset: Breast Cancer ({X_cancer.shape[0]} samples, {X_cancer.shape[1]} features)")

In [ ]:
print("\n--- Approach 1: PCA from Scratch using Eigen Decomposition ---")

class PCAScratch:
    def __init__(self, n_components):
        self.n_components = n_components
        
    def fit(self, X):
        # Center the data
        self.mean = np.mean(X, axis=0)
        X_centered = X - self.mean
        
        # Compute covariance matrix
        self.covariance_matrix = np.cov(X_centered.T)
        
        # Compute eigenvalues and eigenvectors
        eigenvalues, eigenvectors = np.linalg.eig(self.covariance_matrix)
        
        # Sort eigenvalues and eigenvectors in descending order
        idx = eigenvalues.argsort()[::-1]
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]
        
        # Select top n_components
        self.components = eigenvectors[:, :self.n_components]
        self.explained_variance_ratio_ = eigenvalues[:self.n_components] / np.sum(eigenvalues)
        
        return self
    
    def transform(self, X):
        X_centered = X - self.mean
        return np.dot(X_centered, self.components)
    
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

# Apply PCA from scratch
pca_scratch = PCAScratch(n_components=2)
X_pca_scratch = pca_scratch.fit_transform(X_cancer)

# Visualize
plt.figure(figsize=(10, 6))
colors = ['blue', 'red']
markers = ['o', 's']
for i, color, marker in zip([0, 1], colors, markers):
    plt.scatter(X_pca_scratch[y_cancer == i, 0], X_pca_scratch[y_cancer == i, 1],
                c=color, marker=marker, label=f'Class {i}', alpha=0.6)
plt.xlabel('First Principal Component')
plt.ylabel('Second Principal Component')
plt.title('PCA from Scratch on Breast Cancer Dataset')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Explained variance ratio (PC1): {pca_scratch.explained_variance_ratio_[0]:.3f}")
print(f"Explained variance ratio (PC2): {pca_scratch.explained_variance_ratio_[1]:.3f}")
print(f"Total variance explained: {np.sum(pca_scratch.explained_variance_ratio_):.3f}")

In [ ]:
print("\n--- Approach 2: PCA using SVD ---")

class PCASVD:
    def __init__(self, n_components):
        self.n_components = n_components
        
    def fit(self, X):
        # Center the data
        self.mean = np.mean(X, axis=0)
        X_centered = X - self.mean
        
        # Perform SVD
        U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
        
        # Components are the right singular vectors
        self.components = Vt[:self.n_components].T
        
        # Calculate explained variance
        total_var = np.sum(S**2)
        self.explained_variance_ratio_ = (S[:self.n_components]**2) / total_var
        
        return self
    
    def transform(self, X):
        X_centered = X - self.mean
        return np.dot(X_centered, self.components)

# Apply PCA with SVD
pca_svd = PCASVD(n_components=5)
X_pca_svd = pca_svd.fit_transform(X_cancer)

# Compare with sklearn PCA
pca_sklearn = SklearnPCA(n_components=5)
X_pca_sklearn = pca_sklearn.fit_transform(X_cancer)

# Plot cumulative explained variance
plt.figure(figsize=(10, 6))
components_range = range(1, len(pca_svd.explained_variance_ratio_) + 1)
cumulative_var_scratch = np.cumsum(pca_svd.explained_variance_ratio_)
cumulative_var_sklearn = np.cumsum(pca_sklearn.explained_variance_ratio_)

plt.plot(components_range, cumulative_var_scratch, 'bo-', label='SVD PCA (Scratch)', linewidth=2)
plt.plot(components_range, cumulative_var_sklearn, 'ro-', label='Sklearn PCA', linewidth=2, alpha=0.7)
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Cumulative Explained Variance: SVD vs Sklearn')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axhline(y=0.95, color='green', linestyle='--', label='95% Variance Threshold')
plt.legend()
plt.show()

print(f"Variance explained with 5 components (SVD): {cumulative_var_scratch[-1]:.3f}")
print(f"Number of components needed for 95% variance: {np.argmax(cumulative_var_scratch >= 0.95) + 1}")

In [ ]:
print("\n--- Approach 3: Incremental PCA for Large Datasets ---")
from sklearn.decomposition import IncrementalPCA

# Generate large dataset
np.random.seed(42)
X_large = np.random.randn(10000, 50)  # 10,000 samples, 50 features

# Compare standard PCA vs Incremental PCA
n_components = 10

# Standard PCA (requires all data in memory)
print("Standard PCA: Loading all data into memory...")
pca_standard = SklearnPCA(n_components=n_components)
X_pca_standard = pca_standard.fit_transform(X_large)

# Incremental PCA (processes data in batches)
print("Incremental PCA: Processing in batches...")
batch_size = 500
ipca = IncrementalPCA(n_components=n_components, batch_size=batch_size)

# Process data in batches
for i in range(0, len(X_large), batch_size):
    batch = X_large[i:i+batch_size]
    ipca.partial_fit(batch)

X_ipca = ipca.transform(X_large)

# Compare results
print(f"\nStandard PCA components shape: {pca_standard.components_.shape}")
print(f"Incremental PCA components shape: {ipca.components_.shape}")

# Calculate correlation between projections
correlation = np.corrcoef(X_pca_standard[:, 0], X_ipca[:, 0])[0, 1]
print(f"Correlation between first components: {correlation:.4f}")

# Plot comparison of first two components
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X_pca_standard[:, 0], X_pca_standard[:, 1], alpha=0.5, s=10)
axes[0].set_title('Standard PCA')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

axes[1].scatter(X_ipca[:, 0], X_ipca[:, 1], alpha=0.5, s=10)
axes[1].set_title('Incremental PCA (Batch Size=500)')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
plt.tight_layout()
plt.show()

print("\nIncremental PCA advantages:")
print("- Processes data in batches, reducing memory usage")
print("- Ideal for streaming data or datasets too large for memory")
print("- Can handle online learning scenarios")

In [ ]:
print("\n" + "="*80)
print("ALGORITHM 3: K-NEAREST NEIGHBORS (KNN)")
print("="*80)

# Load dataset for KNN
from sklearn.datasets import load_wine
wine = load_wine()
X_wine = wine.data
y_wine = wine.target
feature_names_wine = wine.feature_names

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_wine, y_wine, test_size=0.3, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Dataset: Wine ({X_wine.shape[0]} samples, {X_wine.shape[1]} features, 3 classes)")
print(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}")

In [ ]:
print("\n--- Approach 1: KNN from Scratch with Euclidean Distance ---")

class KNNScratch:
    def __init__(self, k=3, distance_metric='euclidean'):
        self.k = k
        self.distance_metric = distance_metric
        
    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
        
    def _euclidean_distance(self, x1, x2):
        return np.sqrt(np.sum((x1 - x2)**2))
    
    def _manhattan_distance(self, x1, x2):
        return np.sum(np.abs(x1 - x2))
    
    def _predict_single(self, x):
        # Compute distances to all training points
        if self.distance_metric == 'euclidean':
            distances = [self._euclidean_distance(x, x_train) for x_train in self.X_train]
        else:
            distances = [self._manhattan_distance(x, x_train) for x_train in self.X_train]
        
        # Get k nearest neighbors
        k_indices = np.argsort(distances)[:self.k]
        k_nearest_labels = [self.y_train[i] for i in k_indices]
        
        # Return most common label
        return np.bincount(k_nearest_labels).argmax()
    
    def predict(self, X):
        predictions = [self._predict_single(x) for x in X]
        return np.array(predictions)
    
    def score(self, X, y):
        predictions = self.predict(X)
        return accuracy_score(y, predictions)

# Test different k values
k_values = [1, 3, 5, 7, 9, 11]
accuracies = []

for k in k_values:
    knn_scratch = KNNScratch(k=k)
    knn_scratch.fit(X_train_scaled, y_train)
    accuracy = knn_scratch.score(X_test_scaled, y_test)
    accuracies.append(accuracy)
    print(f"K={k}: Accuracy = {accuracy:.3f}")

# Best k
best_k = k_values[np.argmax(accuracies)]
print(f"\nBest K value: {best_k} with accuracy {max(accuracies):.3f}")

# Plot accuracy vs k
plt.figure(figsize=(8, 5))
plt.plot(k_values, accuracies, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Number of Neighbors (K)')
plt.ylabel('Accuracy')
plt.title('KNN from Scratch: Accuracy vs K Value')
plt.grid(True, alpha=0.3)
plt.xticks(k_values)
plt.show()